
$\qquad$ $\qquad$$\qquad$  **TDA231/DIT370 Discrete Optimization: Home Assignment 3 -- LP Duality and the Primal-Dual Algorithm** <br />
$\qquad$ $\qquad$$\qquad$                   **During grading time, direct queries regarding Q-1,2 to David & Q-3 to Marc** <br />
$\qquad$ $\qquad$$\qquad$                     **Due Date: 03/03/2025** <br />
$\qquad$ $\qquad$$\qquad$                   **Submitted by: Elias Söderberg, 20030507-8792., elisod@chalmers.se** <br />
$\qquad$ $\qquad$$\qquad$                   **Submitted by: Jonathan Sunnerstam, 20020420-6734., jonsun@chalmers.se** <br />



---


General guidelines:
*   All solutions to theoretical and pratical problems must be submitted in this ipynb notebook and equations, wherever required, should be formatted using LaTeX math-mode.
*   All discussion regarding practical problems, along with solutions and plots should be specified in this notebook. All plots/results should be visible such that the notebook does not have to be run. But the code in the notebook should reproduce the plots/results if we choose to do so.
*   Your name, personal number and email address should be specified above.
*   All tables and other additional information should be included in this notebook.
*   Before submitting, make sure that your code can run on another computer. That all plots can show on another computer including all your writing. It is good to check if your code can run here: https://colab.research.google.com.


# Problem 1.

Consider the following LP problem:

\begin{alignat*}{2}
\max \ &4x_1-2x_2+5x_3+6x_4+7x_5\\
\\
\textrm{s.t} \quad
&2x_1 + 2x_2 - 4x_3 + 4x_4 + 8x_5 &&\leq 6\\
&2x_1 + \ \ {}x_2 - 2x_3 - \ \ x_4 - 3x_5 &&\geq -1\\
&5x_1 - 2x_2 + 4x_3 + 4x_4 + 2x_5 &&= 5\\
&2x_1 - 2x_2 + 5x_3 + 3x_4 + \ \ x_5 &&\leq 4\\
&\hspace{5.3cm} \vec x &&\geq \vec 0
\end{alignat*}

* (4 points) Write the LP dual of this problem.
* (3 points) Use CVXPY to compute the primal and dual optimum solutions and compare their values.
* (3 points) Check the complementary slackness conditions.

The LP dual of the problem can be found using the "recipe" described in chapter 6.2.

Let $y_1, y_2, y_3, y_4$ be the respective dual variables of the constraints:
$$
\begin{aligned}
& y_1 \geq 0 \\
& y_2 \leq 0 \\
& y_3 \in \mathbb{R} \\
& y_4 \geq 0 
\end{aligned}
$$

The full LP dual thus becomes:
$$
\begin{aligned}
\text{min} \quad & 6y_1 - y_2 + 5y_3 + 4y_4 \\
\text{s.t.} \quad & 2y_1 + 2y_2 + 5y_3 + 2y_4 \ge 4 \\
& 2y_1 + y_2 - 2y_3 - 2y_4 \ge -2 \\
& -4y_1 -2y_2 + 4y_3 + 3y_4 \ge 6 \\
& 8y_1 - 3y_2 + 2y_3 + y_4 \ge 7 \\
& y_1 \ge 0, y_2 \le 0, y_3 \in \mathbb{R}, y_4 \ge 0
\end{aligned}
$$

In [ ]:
import numpy as np 
import cvxpy as cp

def primal():
    x = cp.Variable(5, nonneg=True)
    c = [4, -2, 5, 6, 7]

    A = [
        [2, 2, -4, 4, 8],
        [2, 1, -2, -1, -3],
        [5, -2, 4, 4, 2],
        [2, -2, 5, 3, 1],
    ]

    constraints = [
        (A[0] @ x) <= 6,
        (A[1] @ x) >= -1,
        (A[2] @ x) == 5,
        (A[3] @ x) <= 4,

    ]

    obj = cp.Maximize(x @ c)
    prob = cp.Problem(obj, constraints)
    prob.solve()

    print("Primal solution:")
    print(f"Optimal Value: {np.round(prob.value, 4)}")

    # for complementary slackness
    return x.value

def dual():
   # cp vector in order to use @
    y = cp.hstack([
        cp.Variable(nonneg=True),
        cp.Variable(nonpos=True),
        cp.Variable(),
        cp.Variable(nonneg=True),
    ])
        
    c = [6, -1, 5, 4]

    # np array to auto shape
    A = np.array([
        [2, 2, 5, 2],
        [2, 1, -2, -2],
        [-4, -2, 4, 5],
        [4, -1, 4, 3,],
        [8, -3, 2, 1],
    ])   
    b = [4, -2, 5, 6, 7]

    constraints = [A @ y >= b]

    obj = cp.Minimize(y @ c)
    prob = cp.Problem(obj, constraints)
    prob.solve()

    print("Dual solution:")
    print(f"Optimal Value: {np.round(prob.value, 4)}")

    # for complementary slackness
    return [_y.value for _y in y]

def slacking(opt, slacks):
    for i, (x, slack) in enumerate(zip(opt, slacks), start=1):
        p = x*slack
        print(f"\tvar{i} * slack{i} = {np.round(p, 5)}")

x_opt = primal()
y_opt = dual()

# complementary slackness conditions:
# constraint slack * variable = 0, for primal/dual dual/primal
# primal dual: y_i * slack_primal_constraint_i = 0
# dual primal: x_j * slack_dual_constraint_j

primal_slacks = [
    6 - (2*x_opt[0] + 2*x_opt[1] - 4*x_opt[2] + 4*x_opt[3] + 8*x_opt[4]),
    (2*x_opt[0] + 1*x_opt[1] - 2*x_opt[2] - 1*x_opt[3] - 3*x_opt[4]) - (-1),
    0,
    4 - (2*x_opt[0] - 2*x_opt[1] + 5*x_opt[2] + 3*x_opt[3] + 1*x_opt[4])
]

dual_slacks = [
    (2*y_opt[0] + 2*y_opt[1] + 5*y_opt[2] + 2*y_opt[3]) - 4,
    (2*y_opt[0] + 1*y_opt[1] - 2*y_opt[2] - 2*y_opt[3]) - (-2),
    (-4*y_opt[0] - 2*y_opt[1] + 4*y_opt[2] + 5*y_opt[3]) - 5,
    (4*y_opt[0] - 1*y_opt[1] + 4*y_opt[2] + 3*y_opt[3]) - 6,
    (8*y_opt[0] - 3*y_opt[1] + 2*y_opt[2] + 1*y_opt[3]) - 7
]

print("Primal slack:")
slacking(x_opt, dual_slacks)

print("Dual slack:")
slacking(y_opt, primal_slacks)

The optimal values are the same, which is in coherence with the strong duality theorem following proposition 6.1.1 in the literature.

As the product of each primal constraint's slack and its corresponding dual 
variable is zero, and the product of each primal variable and its corresponding dual constraint is zero, the complementary slackness conditions hold.

# Problem 2.

Consider the LP problem:
\begin{alignat*}{2}
\max \ &6x_1 - 5x_3\\
\\
\textrm{s.t} \quad
&6x_1 - 3x_2 + x_3 &&= 2\\
&3x_1 + 4x_2 + x_3 &&\leq 5\\
&\ \ x_1 - 7x_2 &&\leq 5\\
&\hspace{2.45cm} x_1 &&\geq 0\\
&\hspace{2.45cm} x_2 &&\leq 0\\
&\hspace{2.45cm} x_3 &&\text{ unrestricted}
\end{alignat*}

* (3 points) Write the LP dual of this problem.
* (4 points) Consider the feasible solution $\vec x = (0,0,2)$ to the primal. Check if this solution is optimal by using the complementary slackness conditions to write down the corresponding dual solution.
* (3 points) Use complementary slackness to check if the primal feasible solution $\vec x = (1,0,-4)$ is optimal.

**LP Dual:**

With $y_1, y_2,$ and $y_3$ as the dual variables for each respective contraint:

$$
\begin{aligned}
    \text{min} \quad & 2y_1 + 5y_2 + 5y_3 \\
    \text{s.t.} \quad & 6y_1 + 3y_2 + y_3 \ge 6 \\
    & -3y_1 + 4y_2 - 7y_3 \le 0 \\
    & y_1 + y_2 = -5 \\
    & y_1 \in \mathbb{R}, y_2 \ge 0, y_3 \ge 0
\end{aligned}
$$

**Checking $\vec{x} = (0, 0, 2)$:**

Plugging $\vec{x} = (0, 0, 2)$ into the primal constraints yields:
$$
\begin{aligned}
    &\text{1.} \quad 6*0 - 3*0 + 2 = 2 \quad &\text{tight}. \\
    &\text{2.} \quad 3*0 + 4*0 + 2 = 2 < 5 \quad &\Rightarrow y_2 = 0.\\
    &\text{3.} \quad 0 - 7*0 = 0 < 5\quad &\Rightarrow y_3 = 0. \\
\end{aligned}
$$

The third dual constraint asserts $y_1 + y_2 = -5$. If $y_2 = 0$, then $y_1 = -5$.
This yields the proposed dual solution $y = (-5, 0, 0)$, but plugging the solution into the first dual constraint gives:
$$
6*-5 + 3*0 + 0 = -30 \ngeq 6.
$$
Thus, the solution is not optimal.

**Checking $\vec{x} = (1, 0, -4)$**

Plugging $\vec{x} = (1, 0, -4)$ into the primal constraints yields:
$$
\begin{aligned}
    &\text{1.} \quad 6*1 - 3*0 - 4 = 2 \quad &\text{tight}. \\
    &\text{2.} \quad 3*1 + 4*0 - 4 = -1 < 5 \quad &\Rightarrow y_2 = 0.\\
    &\text{3.} \quad 1 - 7*0 = 1 < 5\quad &\Rightarrow y_3 = 0. \\
\end{aligned}
$$

This again yields $y_1 = -5$, and as shown above, $\vec{y} = (-5, 0, 0)$ is not a solution to the dual, and thus $\vec{x} = (1, 0, -4)$ not an optimal solution to the LP problem.

# Problem 3.

Consider the primal-dual algorithm for vertex cover discussed in class.
* (4 points) Run the algorithm by hand on the graph in the figure below (from your previous homework). Show the values of the primal and dual variables at each iteration.
* (6 points) Implement the primal-dual algorithm as a python script to compute (approximate) vertex covers and run it for the random graph $G(100,0.1)$ from the previous homework.

<img src="https://tinyurl.com/tsnuz2c" alt="Drawing" style="width: 180px;"/>

If the image does not load, try the direct link: https://tinyurl.com/tsnuz2c